

### <font color='green'>Deep Learning for Fraud Detection in Cryptocurrency Financial Transactions</font>

## Installing and Loading the Packages

In [1]:
%env TF_CPP_MIN_LOG_LEVEL=3

env: TF_CPP_MIN_LOG_LEVEL=3


In [4]:
!{sys.executable} -m pip install tensorflow==2.19.0


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
import sys
print(sys.executable)          # qual Python está sendo usado
print(sys.version)             # versão do Python

import subprocess
result = subprocess.run([sys.executable, '-m', 'pip', 'show', 'tensorflow'], 
                       capture_output=True, text=True)
print(result.stdout)   

/usr/local/bin/python
3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:47) [Clang 13.0.0 (clang-1300.0.29.30)]
Name: tensorflow
Version: 2.19.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, libclang, ml-dtypes, numpy, opt-einsum, packaging, protobuf, requests, setuptools, six, tensorboard, termcolor, typing-extensions, wrapt
Required-by: 



In [6]:
# Imports
import sklearn
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import tensorflow as tf
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras import Input
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
import warnings
warnings.filterwarnings('ignore')

In [7]:
# Load the data
df_fraud = pd.read_csv('dataset/dataset.csv')

In [8]:
# Shape
df_fraud.shape

(9841, 51)

In [9]:
# View the first few lines
df_fraud.head()

,Unnamed: 0,Index,Address,FLAG,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,...,ERC20 min val sent,ERC20 max val sent,ERC20 avg val sent,ERC20 min val sent contract,ERC20 max val sent contract,ERC20 avg val sent contract,ERC20 uniq sent token name,ERC20 uniq rec token name,ERC20 most sent token type,ERC20_most_rec_token_type
0,0,1,0x00009277775ac7d0d59eaad8fee3d10ac6c805e8,0,844.26,1093.71,704785.63,721,89,0,...,0.000000,1.683100e+07,271779.920000,0.0,0.0,0.0,39.0,57.0,Cofoundit,Numeraire
1,1,2,0x0002b44ddb1476db43c868bd494422ee4c136fed,0,12709.07,2958.44,1218216.73,94,8,0,...,2.260809,2.260809e+00,2.260809,0.0,0.0,0.0,1.0,7.0,Livepeer Token,Livepeer Token
2,2,3,0x0002bda54cb772d040f779e88eb453cac0daa244,0,246194.54,2434.02,516729.30,2,10,0,...,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,8.0,NaN,XENON
3,3,4,0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e,0,10219.60,15785.09,397555.90,25,9,0,...,100.000000,9.029231e+03,3804.076893,0.0,0.0,0.0,1.0,11.0,Raiden,XENON
4,4,5,0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89,0,36.61,10707.77,382472.42,4598,20,1,...,0.000000,4.500000e+04,13726.659220,0.0,0.0,0.0,6.0,27.0,StatusNetwork,EOS


In [10]:
# Target variable
df_fraud.FLAG.value_counts()

FLAG
0    7662
1    2179
Name: count, dtype: int64

In [11]:
df_fraud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841 entries, 0 to 9840
Data columns (total 51 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Unnamed: 0                                            9841 non-null   int64  
 1   Index                                                 9841 non-null   int64  
 2   Address                                               9841 non-null   object 
 3   FLAG                                                  9841 non-null   int64  
 4   Avg min between sent tnx                              9841 non-null   float64
 5   Avg min between received tnx                          9841 non-null   float64
 6   Time Diff between first and last (Mins)               9841 non-null   float64
 7   Sent tnx                                              9841 non-null   int64  
 8   Received Tnx                                          9841

## Data Cleansing

In [12]:
df_fraud.head()

,Unnamed: 0,Index,Address,FLAG,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,...,ERC20 min val sent,ERC20 max val sent,ERC20 avg val sent,ERC20 min val sent contract,ERC20 max val sent contract,ERC20 avg val sent contract,ERC20 uniq sent token name,ERC20 uniq rec token name,ERC20 most sent token type,ERC20_most_rec_token_type
0,0,1,0x00009277775ac7d0d59eaad8fee3d10ac6c805e8,0,844.26,1093.71,704785.63,721,89,0,...,0.000000,1.683100e+07,271779.920000,0.0,0.0,0.0,39.0,57.0,Cofoundit,Numeraire
1,1,2,0x0002b44ddb1476db43c868bd494422ee4c136fed,0,12709.07,2958.44,1218216.73,94,8,0,...,2.260809,2.260809e+00,2.260809,0.0,0.0,0.0,1.0,7.0,Livepeer Token,Livepeer Token
2,2,3,0x0002bda54cb772d040f779e88eb453cac0daa244,0,246194.54,2434.02,516729.30,2,10,0,...,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,8.0,NaN,XENON
3,3,4,0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e,0,10219.60,15785.09,397555.90,25,9,0,...,100.000000,9.029231e+03,3804.076893,0.0,0.0,0.0,1.0,11.0,Raiden,XENON
4,4,5,0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89,0,36.61,10707.77,382472.42,4598,20,1,...,0.000000,4.500000e+04,13726.659220,0.0,0.0,0.0,6.0,27.0,StatusNetwork,EOS


In [13]:
# Adjust the name to lowercase
df_fraud.columns = [x.lower() for x in df_fraud.columns]

In [14]:
# remove
cols_to_drop = [' erc20 most sent token type',
                ' erc20_most_rec_token_type',
                'address',
                'index',
                'unnamed: 0']

In [16]:
# Select the attributes by filtering the columns to be removed and the target variable
attr = [x for x in df_fraud.columns if (x != 'flag' and x not in cols_to_drop)]

In [17]:
attr

['avg min between sent tnx',
 'avg min between received tnx',
 'time diff between first and last (mins)',
 'sent tnx',
 'received tnx',
 'number of created contracts',
 'unique received from addresses',
 'unique sent to addresses',
 'min value received',
 'max value received ',
 'avg val received',
 'min val sent',
 'max val sent',
 'avg val sent',
 'min value sent to contract',
 'max val sent to contract',
 'avg value sent to contract',
 'total transactions (including tnx to create contract',
 'total ether sent',
 'total ether received',
 'total ether sent contracts',
 'total ether balance',
 ' total erc20 tnxs',
 ' erc20 total ether received',
 ' erc20 total ether sent',
 ' erc20 total ether sent contract',
 ' erc20 uniq sent addr',
 ' erc20 uniq rec addr',
 ' erc20 uniq sent addr.1',
 ' erc20 uniq rec contract addr',
 ' erc20 avg time between sent tnx',
 ' erc20 avg time between rec tnx',
 ' erc20 avg time between rec 2 tnx',
 ' erc20 avg time between contract tnx',
 ' erc20 min val

In [ ]:
# Extract unique values
unique_values = df_fraud.nunique()

In [19]:
unique_values

unnamed: 0                                              9841
index                                                   4729
address                                                 9816
flag                                                       2
avg min between sent tnx                                5013
avg min between received tnx                            6223
time diff between first and last (mins)                 7810
sent tnx                                                 641
received tnx                                             727
number of created contracts                               20
unique received from addresses                           256
unique sent to addresses                                 258
min value received                                      4589
max value received                                      6302
avg val received                                        6767
min val sent                                            4719
max val sent            

In [20]:
# It only retains attributes with more than one unique value (attributes that are not constants)
attr = [x for x in attr if x in unique_values.loc[(unique_values > 1)]]

In [21]:
df_fraud[attr].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841 entries, 0 to 9840
Data columns (total 38 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   avg min between sent tnx                              9841 non-null   float64
 1   avg min between received tnx                          9841 non-null   float64
 2   time diff between first and last (mins)               9841 non-null   float64
 3   sent tnx                                              9841 non-null   int64  
 4   received tnx                                          9841 non-null   int64  
 5   number of created contracts                           9841 non-null   int64  
 6   unique received from addresses                        9841 non-null   int64  
 7   unique sent to addresses                              9841 non-null   int64  
 8   min value received                                    9841

In [26]:
# Defining a custom class that inherits from BaseEstimator and TransformerMixin
class fraudPipeSteps(BaseEstimator, TransformerMixin):

    # Constructor method to initialize the class with a list of columns
    def __init__(self, columns=[]):
        
        # Assigning the `columns` argument to the `self.columns` instance attribute
        self.columns = columns

    # The fit method is used to adjust (train) the transformation in the training data
    def fit(self, X, y = None):

        return self

    # The transform method is used to transform the input data
    def transform(self, X):

        X = X.copy()

        return X

In [27]:
# Defining a class that inherits from fraudPipeSteps
class fraudSelectColumns(fraudPipeSteps):

    # The transform method is used to transform the input data
    def transform(self, X):

        X = X.copy()

        return X[self.columns]

In [28]:
# Defining a class that inherits from fraudPipeSteps
class fraudFillInData(fraudPipeSteps):

    # Fit method to adjust the transformation in training data
    def fit(self, X, y = None):
        
        # It calculates the average of each column specified in self.columns and stores it in the self.means dictionary
        self.means = { col: X[col].mean() for col in self.columns }

        return self

    # The transform method is used to transform the input data
    def transform(self, X):

        X = X.copy()
        
        # It iterates over each column specified in self.columns
        for col in self.columns:
            
            # Fill in missing values ​​in the column with the average calculated in the adjustment phase
            X[col] = X[col].fillna(self.means[col])
        
        # Returns the transformed data
        return X

In [29]:
# Defining a class that inherits from fraudPipeSteps
class StandardizedDataFraud(fraudPipeSteps):

    # Fit method to adjust the scaler to the training data
    def fit(self, X, y = None):
        
        # Initializes a StandardScaler instance to standardize the data
        self.scaler = StandardScaler()
        
        # Adjusts the scaler on the columns specified in self.columns
        self.scaler.fit(X[self.columns])

        return self

    # The transform method is used to transform the input data
    def transform(self, X):
        
        X = X.copy()

        X[self.columns] = self.scaler.transform(X[self.columns])

        return X

In [30]:
# Defining a class that inherits from fraudPipeSteps
class fraudGetData(fraudPipeSteps):

    # The transform method is used to transform the input data
    def transform(self, X):
        
        X = X.copy()
        
        # Returns the values ​​from the DataFrame as a NumPy array
        return X.values

In [31]:
# Create the pipeline
fraud_pipe_preprocessing = Pipeline([('feature_selection', fraudSelectColumns(attr)),
                                       ('fill_missing', fraudFillInData(attr)),
                                       ('standard_scaling', StandardizedDataFraud(attr)),
                                       ('returnValues', fraudGetData())])

In [32]:
# Input variables
X = df_fraud[attr]

In [33]:
# Output variable
y = df_fraud['flag']

In [34]:
# Adjust the type of the output variable
y = to_categorical(y)

In [36]:
# Divide the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.30, random_state = 42)

In [37]:
# Standardize the data
X_train = fraud_pipe_preprocessing.fit_transform(X_treino)
X_test = fraud_pipe_preprocessing.transform(X_teste)

In [38]:
X_train

array([[-0.23259911, -0.35751318, -0.67960935, ..., -0.01331307,
        -0.21172682, -0.2313969 ],
       [-0.23259911,  0.0138933 , -0.06181513, ..., -0.01331307,
        -0.21172682, -0.17365057],
       [-0.23259911, -0.34333968, -0.67254633, ..., -0.01331307,
        -0.21172682, -0.2313969 ],
       ...,
       [-0.23259911, -0.35751318, -0.67960389, ..., -0.01331307,
        -0.21172682, -0.28914324],
       [-0.22473065, -0.35751318, -0.67858957, ..., -0.01331307,
        -0.21172682, -0.28914324],
       [-0.23250192, -0.35750828, -0.67958981, ..., -0.01331307,
        -0.21172682, -0.28914324]])

In [39]:
X_test

array([[-1.54379550e-01, -2.63825638e-01,  3.32086109e-01, ...,
        -1.33130727e-02, -2.11726819e-01, -4.11559255e-04],
       [-9.86466733e-02, -3.20183744e-01, -6.49312398e-01, ...,
        -1.33130727e-02, -2.11726819e-01, -2.31396905e-01],
       [-2.25103708e-01, -3.57513182e-01, -6.78637885e-01, ...,
        -1.33130727e-02, -2.11726819e-01, -2.89143241e-01],
       ...,
       [-2.32514779e-01, -3.57494918e-01, -6.79590458e-01, ...,
        -1.33130727e-02, -2.11726819e-01, -2.89143241e-01],
       [-2.32599109e-01, -3.57233873e-01, -6.75564107e-01, ...,
        -1.33130727e-02, -2.11726819e-01, -2.31396905e-01],
       [-2.31476133e-01, -3.57513182e-01, -6.79391008e-01, ...,
        -1.33130727e-02, -2.11726819e-01, -2.89143241e-01]])

## Building the Deep Learning Model

In [40]:
# Create a sequence of layers
model_fraud = Sequential()

In [41]:
# Adds an input layer to the model with the shape specified by the length of 'attributes'
model_fraud.add(Input(shape = (len(attr),)))

# Adds a dense layer to the model with 'len(attributes)' units and the 'relu' activation function
model_fraud.add(Dense(len(attr), activation = 'relu'))

# Adds a dense layer to the model with 20 units and the 'relu' activation function
model_fraud.add(Dense(20, activation = 'relu'))

# Adds a dense layer to the model with 5 units and the 'relu' activation function
model_fraud.add(Dense(5, activation = 'relu'))

# Adds a dense layer to the model with 2 units and the 'softmax' activation function (output layer)
model_fraud.add(Dense(2, activation = 'softmax'))

In [42]:
# Model compilation
model_fraud.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

In [43]:
# Summary
model_fraud.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 38)             │         1,482 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           780 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           105 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,379 (9.29 KB)

 Trainable params: 2,379 (9.29 KB)

 Non-trainable params: 0 (0.00 B)

## Training and Evaluation of the Model

In [44]:
%%time
model_fraud.fit(X_train, y_train, validation_data = (X_test, y_test), epochs = 10)

Epoch 1/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 1s 976us/step - accuracy: 0.7737 - loss: 0.4522 - val_accuracy: 0.7795 - val_loss: 0.3728
Epoch 2/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 0s 589us/step - accuracy: 0.8399 - loss: 0.3427 - val_accuracy: 0.8750 - val_loss: 0.3137
Epoch 3/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 0s 649us/step - accuracy: 0.8869 - loss: 0.2861 - val_accuracy: 0.9241 - val_loss: 0.2802
Epoch 4/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 0s 657us/step - accuracy: 0.9217 - loss: 0.2486 - val_accuracy: 0.9424 - val_loss: 0.2510
Epoch 5/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 0s 613us/step - accuracy: 0.9368 - loss: 0.2206 - val_accuracy: 0.9397 - val_loss: 0.2323
Epoch 6/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 0s 612us/step - accuracy: 0.9456 - loss: 0.1906 - val_accuracy: 0.9458 - val_loss: 0.2030
Epoch 7/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 0s 610us/step - accuracy: 0.9492 - loss: 0.1662 - val_accuracy: 0.9485 - val_loss: 0.1959
Epoch 8/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 0s 577us/step - accuracy: 0.9528 - loss: 0.1524 - 

In [45]:
# Predictions based on test data
test_predictions = [np.argmax(x) for x in model_fraud.predict(X_test)]

93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 451us/step


In [46]:
# Calculate the Accuracy
acc = metrics.accuracy_score(test_predictions, [np.argmax(y) for y in y_test])

Accuracy is a metric for evaluating classification models. It represents the proportion of correct predictions made by the model relative to the total number of samples evaluated. In other words, it is the number of correct predictions divided by the total number of predictions made, indicating how well the model is correctly classifying the samples.

In [47]:
print(f'Accuracy in Test Data: {acc:,.2%}')

Accuracy in Test Data: 95.26%


In [48]:
# Calculates AUC
auc = metrics.roc_auc_score([np.argmax(y) for y in y_test], model_fraud.predict(X_test)[:,1])

93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 343us/step


The Area Under the Curve (AUC) is a metric used to evaluate the performance of binary classification models. It is based on the ROC (Receiver Operating Characteristic) curve, which is a graph showing the relationship between the true positive rate (sensitivity) and the false positive rate (1 - specificity) for different decision thresholds.

The AUC represents the total area under the ROC curve and can range from 0 to 1. An AUC of 0.5 indicates random performance, while an AUC of 1.0 indicates perfect classification. Therefore, the closer the AUC value is to 1, the better the model's ability to distinguish between positive and negative classes.

In [49]:
print(f'AUC in Test Data - {auc:,.2%}')

AUC in Test Data - 97.60%


## Deployment of the Model and Fraud Detection in New Cryptocurrency Transactions

In [50]:
# Loads the new data for a transaction
dataset_new = pd.read_csv('dataset/new_dataset.csv')

In [ ]:
dataset_new

,avg min between sent tnx,avg min between received tnx,time diff between first and last (mins),sent tnx,received tnx,number of created contracts,unique received from addresses,unique sent to addresses,min value received,max value received,...,erc20 uniq sent addr.1,erc20 uniq rec contract addr,erc20 min val rec,erc20 max val rec,erc20 avg val rec,erc20 min val sent,erc20 max val sent,erc20 avg val sent,erc20 uniq sent token name,erc20 uniq rec token name
0,2570.59,3336.01,30572.7,8,3,0,2,4,0.1,40.0,...,0.0,1.0,600.0,600.0,600.0,0.0,0.0,0.0,0.0,1.0


In [52]:
# It applies the same pipeline used for the training data
new_transformed_data = fraud_pipe_preprocessing.transform(dataset_new)

In [53]:
# Data in the format that the model expects to receive
new_transformed_data

array([[-0.11012514, -0.20890417, -0.5852175 , -0.14846488, -0.17435227,
        -0.02949086, -0.10071957, -0.08219271, -0.12770293, -0.03945017,
        -0.0281488 , -0.04686515, -0.05420181, -0.110975  ,  0.        ,
        -0.01204994, -0.01204994, -0.20568049, -0.02975031, -0.03630726,
        -0.01204994, -0.0197842 , -0.08228942, -0.06411176, -0.01425745,
        -0.02258837, -0.05367989, -0.09036273, -0.05319145, -0.22783703,
         0.00627018, -0.05510854, -0.02401607, -0.01364824, -0.0135741 ,
        -0.01331307, -0.21172682, -0.2313969 ]])

In [54]:
# Extract the most likely prediction
prev = [np.argmax(x) for x in model_fraud.predict(new_transformed_data)]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


In [55]:
type(prev)

list

In [56]:
# Result
if prev[0] == 0:
    print("According to the model, this transaction does not represent fraud.")
else:
    print("According to the model, this transaction may represent fraud. Trigger human verification!")

According to the model, this transaction does not represent fraud.
